# Setup: put the three CSV files into PostgreSQL
Run every cell once, top to bottom. Kernel: **SQL Pandas FaceOff**.
This creates three tables (applications, products, submissions) in your `fda_drugs` database.

In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

DATA = Path("..") / "data"
engine = create_engine("postgresql+psycopg2:///fda_drugs")

## 1. Create empty tables
The column types are chosen on purpose: codes such as `000793` stay as text so the leading zeros are not lost.

In [2]:
ddl = """
DROP TABLE IF EXISTS submissions;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS applications;

CREATE TABLE applications (
    appl_no      text,
    appl_type    text,
    sponsor_name text
);

CREATE TABLE products (
    appl_no           text,
    product_no        text,
    drug_name         text,
    active_ingredient text,
    form              text,
    strength          text
);

CREATE TABLE submissions (
    appl_no           text,
    submission_type   text,
    submission_no     integer,
    submission_status text,
    status_date       date,
    review_priority   text
);
"""
with engine.begin() as conn:
    conn.exec_driver_sql(ddl)
print("tables created")

tables created


## 2. Load the CSV files
We read everything as text first, then convert the two columns that are not text.

In [3]:
apps = pd.read_csv(DATA / "applications.csv", dtype=str)
products = pd.read_csv(DATA / "products.csv", dtype=str)
subs = pd.read_csv(DATA / "submissions.csv", dtype=str)

subs["submission_no"] = subs["submission_no"].astype(int)
subs["status_date"] = pd.to_datetime(subs["status_date"]).dt.date

with engine.begin() as conn:
    apps.to_sql("applications", conn, if_exists="append", index=False)
    products.to_sql("products", conn, if_exists="append", index=False)
    subs.to_sql("submissions", conn, if_exists="append", index=False)
print(len(apps), len(products), len(subs))

360 599 2526


## 3. Check
You should see 360, 599 and 2526.

In [4]:
with engine.connect() as conn:
    for t in ["applications", "products", "submissions"]:
        print(t, conn.execute(text(f"SELECT count(*) FROM {t}")).scalar())

applications 360
products 599
submissions 2526
